- https://verl.readthedocs.io/en/latest/examples/config.html

In [1]:
from IPython.display import Image

- 带 kl 散度惩罚的强化学习目标

$$
\mathcal{J}(\pi_\theta) = \mathbb{E}_{\mathbf{q} \sim p_Q} \left[ \mathbb{E}_{\mathbf{o} \sim \pi_\theta(\cdot|\mathbf{q})} [R(\mathbf{q}, \mathbf{o})] - \beta D_{KL}[\pi_\theta(\cdot|\mathbf{q}) || \pi_{\text{ref}}(\cdot|\mathbf{q})] \right]
$$

- PPO-Clip （代理）目标函数

$$
\mathcal{J}_{\text{PPO}}(\pi_\theta) = \mathbb{E}_{\mathbf{q} \sim p_Q, \mathbf{o} \sim \pi_{\theta_{\text{old}}}(\cdot|\mathbf{q})} \left[ \sum_{t=1}^{|\mathbf{o}|} \min \left[ \frac{\pi_\theta(o_t|\mathbf{q}, \mathbf{o}_{<t})}{\pi_{\theta_{\text{old}}}(o_t|\mathbf{q}, \mathbf{o}_{<t})} \hat{A}_t, \text{clip}\left(\frac{\pi_\theta(o_t|\mathbf{q}, \mathbf{o}_{<t})}{\pi_{\theta_{\text{old}}}(o_t|\mathbf{q}, \mathbf{o}_{<t})}, 1-\epsilon, 1+\epsilon\right) \hat{A}_t \right] \right]
$$

- training 的时候，关注的是 rewards curve，而不是 loss curve
    - ppo-clip 的 loss 是一种代理目标，保守的、低方差的代理目标（surrogate）
    - ppo-clip
        - 当 A_t > 0 (好的动作) 时，r_t(θ) (新旧策略概率比) 不会被允许超过 1+ε。这意味着，即使我们发现了一个超好的动作，我们也不会过于激进地增加选择它的概率，防止单步更新过猛。
        - 当 A_t < 0 (差的动作) 时，r_t(θ) 不会被允许低于 1-ε。这意味着，我们也不会过于激进地去惩罚一个坏动作。
    - 关于 clip fraction
        - 高 clip fraction ⇒ 有效梯度变少
            - 被裁剪样本不给梯度，等价于“有效 batch 变小”。clip fraction 越高，更新越被少数未裁剪样本主导，学习容易停滞或极不稳定。
        - 过低 clip fraction ⇒ 步子太小
        - 与 kl 的耦合
            - 高 clip fraction ＋ 高 KL：步子过猛（LR 大、epoch 多、RM/优势刻度过大）。
            - 高 clip fraction 但 KL 不高：多半是同一批数据被反复训练把 $r_t$ 推到边界（epoch 太多 / 复用过度）。
- 除了 rewards curve 还应该关注什么
    - actor entropy (generation entropy)：探索 or 很快收敛
    - kl reg
    - clip fraction：0.1-0.4
        - 过高＝过猛；长期 0＝步太小
    - adv 等

### customizing verl

- 搜索 `.backward()`
    - `loss.backward()`
- 修改 `compute_policy_loss`: `core_algos.py` 注册不同的 pg_loss
    - 比如添加 `entropy_loss`
- 如果要改 training logic，则需要自定义 Trainer
    - `class RayDAPOTrainer(RayPPOTrainer):`：只是覆盖了 `fit` 函数（为了实现 dynamic sampling）

- verl/trainer/ppo/core_algos.py
    - @register_policy_loss("vanilla")
    - @register_policy_loss("gpg")
    - @register_policy_loss("kl_cov")
    - @register_policy_loss("clip_cov")
    - @register_policy_loss("geo_mean")
    - @register_policy_loss("gspo")

### actor & critic

In [2]:
# single controller
Image(url='./figs/single-controller.png', width=400)

- 注意区分奖励（reward model）和价值（critic model）
    - token级别的价值（由 Critic 网络提供）：$V(s_t)$
    - token级别的奖励：$r_t=r_{RM}-\beta\log\frac{\pi_\theta(a_t|s_t)}{\pi_{\text{SFT}}(a_t|s_t)}$
        - $r_{RM}: $通常只在序列的最后一个Token处有一个非零值（由reward model 提供），其余时间步为0。
- 优势 $\hat A_t$ 的计算
    - TD error: $\delta_t=r_t+\gamma V(s_{t+1}-V(s_t))$
    - PPO GAE: $\hat{A}_t = \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + \cdots + (\gamma\lambda)^{T-t-1}\delta_{T-1}$

## pg loss

In [3]:
Image(url='https://huggingface.co/blog/assets/93_deep_rl_ppo/recap.jpg', width=400)

https://huggingface.co/blog/deep-rl-ppo
$$
L^{PPO}=E_{t}[\max(-r_t(\theta)\hat A_t, -\text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat A_t)]
$$

- 当 $\hat A_t\gt 0$（好动作）时，不希望 $r_t(\theta)$ 过大，将其限制在 $1+\epsilon$ 之内
    - $r_t(\theta) > 1+\epsilon$
    - $\pi_\theta(a_t|s_t) >> \pi_{\theta_{old}}(a_t|s_t)$
- 当 $\hat A_t\lt 0$（坏动作）时，不希望 $r_t(\theta)$ 过小，将其限制在 $1-\epsilon$ 之内
    - $r_t(\theta) < 1-\epsilon$
- 还需要注意 gradient 的部分；
    - 什么情况下不学习（不更新模型）
    - 方向盘（梯度） vs. 放大器
- sign of objective is sign of $\hat A_t$
    - 正 objective：$\hat A_t\gt 0$，增大 $\pi_\theta(a_t|s_t)$
    - 负 objective：$\hat A_t\lt 0$，降低 $\pi_\theta(a_t|s_t)$
- $\hat A_t$: 不贡献梯度（是一个不在计算图中的标量）
    - $\hat A_t$ 的计算来源于 $\pi_{\theta_{old}}$（$\pi_{\theta_{old}$ 在一次 ppo epoch 中只负责 generation 与采集数据）

### pg loss curve

- policy_loss 的正负直接反映了在一个训练批次（batch）中，智能体所采取的行动的平均质量（相对于价值函数的评估）。
- policy_loss < 0 (负数)代表什么？
    - 这意味着 L_CLIP(θ) 是正数。
    - 深层含义？ 在这个批次中，“好”的动作（$\hat A_t>0$）所带来的正面影响，超过了“坏”的动作（$\hat A_t < 0$）带来的负面影响。简单来说，智能体在这个批次中的平均表现超出了它自己的预期（价值函数 $V(s_t)$）。
优化方向？ 优化器会试图最小化这个负数，即让它变得更负。这会进一步增强那些“好”动作的概率。
    - 是好是坏？
        - 通常是好兆头：表明智能体正在有效地探索并找到了能带来更高回报的策略。这是学习在正常发生的信号。
    - 潜在风险：如果 policy_loss 持续保持在非常大的负值，可能意味着价值函数（Critic）的更新跟不上策略（Actor）的提升。Critic 总是低估当前策略的价值，导致 Advantage 持续偏高。这可能预示着潜在的不稳定。
- policy_loss > 0 (正数)
    - 代表什么？ 这意味着 L_CLIP(θ) 是负数。
    - 深层含义？ 在这个批次中，“坏”的动作所带来的负面影响，超过了“好”的动作带来的正面影响。智能体的平均表现低于它自己的预期。
    - 优化方向？ 优化器会试图最小化这个正数，即让它向 0 靠近。这会削弱那些“坏”动作的概率。
    - 是好是坏？
        - 这是学习的正常组成部分：智能体需要通过试错来学习。采取了坏的动作，然后通过正的 policy_loss 来惩罚这些动作，这是完全正常的。
        - 危险信号：如果 policy_loss 持续保持在较高的正值，这是一个非常糟糕的信号。它意味着智能体在不断地做出比自己预期还要差的决策，策略可能正在恶化或完全没有学到东西。
- 理想的趋势：在 0 附近震荡，无明显上升或下降趋势
    - 一个完美的策略和价值函数组合，意味着对于任何状态，价值函数都能准确预测期望回报。因此，任何动作的优势函数 $\hat A_t$ 都会趋近于 0。
    - 在实际训练中，策略和价值函数是交替迭代、相互追赶的。策略稍微变好一点 (policy_loss 变负)，价值函数马上学习跟上，将 policy_loss 拉回到 0 附近。策略尝试了坏动作 (policy_loss 变正)，然后修正自己，又回到 0 附近。
    - 稳定学习：Actor 和 Critic 步调一致，策略在信赖域内被稳定地优化。
    - 有效基线：价值函数提供了一个准确的基线（baseline），使得优势函数的估计是有意义的。

### dual-clip ppo

In [3]:
Image(url='./figs/dual-clip.png', width=300)

- Dual-Clip PPO
    - https://arxiv.org/pdf/1912.09729
 
$$
L^{\text{dual-clip}}=\min(L^{PPO}, -c\cdot \hat A_t)
$$

### entropy, log_prob

- actor
    - entropy_coeff: 0.0 (default)
```python
# entropy = verl_F.entropy_from_logits(logits)  # (bsz, response_length)
policy_loss = pg_loss - entropy_loss * entropy_coeff
```

- 对于一个多分类问题，给定 logits $L_i$，样本属于类别 $j$ 的概率 $P_j$ 是通过 Softmax 函数计算的：
    - $P(j|L_i)=\frac{e^{L_{ij}}}{\sum_{k=1}^Ve^{L_{ik}}}$
    - 取 log，$\log P(j|L_i)=\log e^{L_{ij}}-\log\sum_{k=1}^Ve^{L_{ik}}=L_{ij}-\text{logsumexp}$

```python
def entropy_from_logits(logits: torch.Tensor):
    """Calculate entropy from logits."""
    pd = torch.nn.functional.softmax(logits, dim=-1)
    entropy = torch.logsumexp(logits, dim=-1) - torch.sum(pd * logits, dim=-1)
    return entropy
```
- logits: `[bz, seq_len, vocab_sz]`
- pd: `[bz, seq_len, vocab_sz]`
- logsumexp: `[bz, seq_len]`
- sum(pd*logits): `[bz, seq_len]`
- 最终的输出：`[bz, seq_len]`
    - 即也是 token 粒度的，也需要经过 loss agg；

In [1]:
import torch
def entropy_from_logits(logits: torch.Tensor):
    """Calculate entropy from logits."""
    pd = torch.nn.functional.softmax(logits, dim=-1)
    entropy = torch.logsumexp(logits, dim=-1) - torch.sum(pd * logits, dim=-1)
    return entropy
    
logits = torch.randn(32, 128, 50000)  # [batch, seq_len, vocab]
entropy = entropy_from_logits(logits)  # [32, 128]
entropy.shape

torch.Size([32, 128])

$$
H=-\sum_vp(v)\log p(v)
$$

- 在 llm 中，就是 generation 生成序列的每个位置 $\pi_\theta(\cdot |q, o_{\lt t})$ 都对应一个词表维度的概率分布

$$
p(v)=\frac{\exp(\text{logits}_v)}{\sum_{v'}\exp(\text{logits}_{v'})}=\frac{\exp(\text{logits}_v)}{Z}
$$

- 则有

$$
\log p(v)=\text{logits}_v-\log Z
$$

- 进一步

$$
H=-\sum_v p(v)\log p(v)=-\sum_v p(v)(\text{logits}_v-\log Z)=\log Z - \sum_v p(v)\text{logits}_v
$$

### use_kl_loss

- http://joschu.net/blog/kl-approx.html
```python
policy_loss = policy_loss + kl_loss * self.config.kl_loss_coef
```

### agg_loss

- $L\in R^{B\times T}$ 表示 loss mat, $M\in {\{0,1\}}^{B\times T}$ 表示损失掩码 (loss_mask)（为 1 表示损失计算在内）
    - $ \mathcal{L}_{\text{token-mean}} = \frac{\sum_{i=1}^{B} \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{\sum_{i=1}^{B} \sum_{j=1}^{T} M_{i,j}} $
    - $\mathcal{L}_{\text{seq-mean-token-sum}} = \frac{1}{B} \sum_{i=1}^{B} \left( \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j} \right)$
    - $\mathcal{L}_{\text{seq-mean-token-mean}} = \frac{1}{B} \sum_{i=1}^{B} \left( \frac{\sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{\sum_{j=1}^{T} M_{i,j}} \right)$
    - $\mathcal{L}_{\text{seq-mean-token-sum-norm}} = \frac{\sum_{i=1}^{B} \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{T} $

```python
# "token-mean"
s = masked_sum(values, mask, axis)
s / (mask.sum(axis=axis) + 1e-8)
```
- https://verl.readthedocs.io/en/latest/algo/grpo.html
    - The original GRPO paper takes the sample-level loss (`seq-mean-token-mean`), which may be unstable in long-CoT scenarios.
    - DrGRPO: `seq-mean-token-sum-norm”`
    - DAPO：`token-mean`
        - https://verl.readthedocs.io/en/latest/algo/dapo.html#flexible-loss-aggregation-mode-token-level-loss
        - Setting `loss_agg_mode` to `token-mean` will mean the (policy gradient) loss across all the tokens in all the sequences in a mini-batch.

### details

- $\pi_{\theta_{\text{old}}}$ 在 training 时计算

```python
# ray_trainer.py, fit 
# recompute old_log_probs
with marked_timer("old_log_prob", timing_raw, color="blue"):
    old_log_prob = self.actor_rollout_wg.compute_log_prob(batch)
```